# Detección y Medición del Ciclo Diurno en Ciclones Tropicales o Subtropicales

Este cuaderno JupyterLab se centra en la detección y cuantificación del ciclo diurno en ciclones tropicales o subtropicales. Comprender estos ciclos es fundamental para mejorar los modelos de pronóstico y entender la dinámica interna de estas tormentas.

Los datos de entrada para este análisis provienen del cuaderno previo, **[05_Calcular_perfil_radial.ipynb](./05_Calcular_perfil_radial.ipynb)**. Específicamente, estamos trabajando con perfiles radiales del campo de **temperatura de brillo (BT)**, una medida clave obtenida de observaciones satelitales que nos permite inferir características de las nubes y la convección dentro del huracán. Estos perfiles han sido cuidadosamente centrados en la tormenta y reproyectados utilizando una **Proyección Acimutal Equidistante** para asegurar que las distancias desde el centro sean precisas, lo cual es crucial para un análisis radial consistente.

La **finalidad principal** de este cuaderno es:
1.  Identificar la presencia del ciclo diurno en el huracán analizado.
2.  Calcular el período de este ciclo a partir de la evolución temporal de la diferencia de perfiles radiales, lo que nos permitirá caracterizar su patrón de variación a lo largo del día.

---

## 1. Preliminares: Configuración del Entorno y Preparación de Datos

### 1.1 Configuración del Entorno de Trabajo y Carga de Configuraciones

Esta sección inicial se dedica a la puesta a punto del entorno de desarrollo. Incluye la carga de las librerías necesarias, la configuración de parámetros específicos para el proyecto y la inicialización de variables globales que serán utilizadas a lo largo de los análisis.

In [ ]:
from goesdl.experimental.utilities import initialize_project, reload_project

settings = initialize_project("../event.yaml", "../settings.yaml")

### 1.2 Carga y Gestión del Inventario de Archivos de Datos

Esta sección se encarga de cargar el **inventario de archivos de datos** necesarios para el análisis. Un inventario de archivos es un registro organizado de las rutas y metadatos temporales de los archivos que contienen la información de temperatura de brillo del huracán Iván.

La función `load_inventory` se utiliza para leer este inventario, permitiendo al cuaderno identificar y acceder a todos los archivos de datos relevantes de manera eficiente. Esto es crucial para asegurar que todos los datos necesarios para construir los perfiles radiales estén disponibles y sean correctamente indexados para su posterior procesamiento.

Como resultado de esta operación, se obtendrán las siguientes variables:
* `dataset_paths`: Una lista o estructura que contiene las rutas completas a cada uno de los archivos de datos brutos.
* `dataset_times`: Una lista o array con los sellos de tiempo correspondientes a cada archivo, lo que permite ordenar cronológicamente la información.

In [ ]:
from goesdl.experimental.utilities import load_inventory

dataset_paths, dataset_times = load_inventory(settings)

### 1.3 Determinación de Parámetros Dinámicos Basados en los Datos

En esta etapa, se calculan y configuran parámetros esenciales para el análisis que **dependen directamente de las características intrínsecas de los datos cargados**. A diferencia de los parámetros fijos definidos en la configuración del proyecto, estos valores se derivan dinámicamente del inventario de archivos (`dataset_paths`) para asegurar que el procesamiento y el análisis se adapten con precisión al conjunto de datos específico que se está utilizando.

La función `get_computed_parameters` es la encargada de realizar estos cálculos. Esto puede incluir, por ejemplo, la determinación de rangos temporales o espaciales óptimos, o la preparación de estructuras de datos que se ajusten a las dimensiones de los datos brutos. El resultado de esta operación se almacena en la variable `parameters`, que encapsula todos estos valores derivados y los pone a disposición para los pasos subsiguientes del cuaderno.

In [ ]:
from goesdl.experimental.utilities import get_computed_parameters

parameters = get_computed_parameters(settings, dataset_paths)

## 2. Ejecución del algoritmo

### 2.1 Paso 1: Generación de series de tiempo

1. Carga cronológica de los perfiles radiales, $P(r)_{t}$ de $T_{bb}$, para cada instante $t$ a analizar
2. Diferencia de perfiles radiales $D(r)_{t} = P(r)_{t} - P(r)_{t+\delta t}$, con una separación temporal, $\delta t$
3. Creación de la serie de tiempo $S(t)_{r} = D(r)_{t}$, para los valores de $r$ dados arriba

## 2. Ejecución del Algoritmo Principal para la Generación de Series Temporales

Esta sección marca el inicio de la fase de procesamiento principal, donde se implementa el algoritmo diseñado para preparar los datos de temperatura de brillo (BT) de los perfiles radiales y transformarlos en series temporales adecuadas para el análisis del ciclo diurno. El objetivo es resaltar las variaciones temporales de la BT en función del radio, que son clave para identificar patrones periódicos.

### 2.1 Paso 1: Generación de Series de Tiempo a partir de Perfiles Radiales

Este primer paso es fundamental para estructurar los datos de una manera que facilite la detección del ciclo diurno. Implica las siguientes sub-etapas:

1.  **Carga Cronológica de Perfiles Radiales ($P(r)_{t}$ de $T_{bb}$):** Se realiza la carga secuencial de los perfiles radiales de temperatura de brillo ($T_{bb}$) para cada instante de tiempo ($t$) disponible. Aquí, $P(r)_t$ representa el perfil radial de la temperatura de brillo en un radio $r$ y en un tiempo $t$. Esta carga asegura que los datos estén ordenados temporalmente para un análisis coherente.

2.  **Cálculo de la Diferencia de Perfiles Radiales ($D(r)_{t}$):** Para realzar las variaciones temporales y mitigar el efecto de componentes estáticas o de muy baja frecuencia, se calcula la diferencia entre perfiles radiales consecutivos o separados por un intervalo $\delta t$. La fórmula es $D(r)_{t} = P(r)_{t} - P(r)_{t+\delta t}$. Esta operación es crucial para aislar las fluctuaciones de corto plazo que son características del ciclo diurno, eliminando la tendencia general o las estructuras espaciales persistentes del ciclón.

3.  **Creación de la Serie de Tiempo ($S(t)_{r}$):** Finalmente, a partir de estas diferencias, se construye una serie de tiempo $S(t)_r = D(r)_t$ para cada valor de radio ($r$) previamente definido. Esto significa que para cada anillo concéntrico alrededor del centro de la tormenta, obtenemos una serie temporal que describe cómo la *diferencia* de la temperatura de brillo evoluciona a lo largo del tiempo. Estas series de tiempo son las que se utilizarán en los análisis espectrales posteriores para identificar la periodicidad del ciclo diurno.

La variable resultante `bt_raw_timeseries` es una **lista de arrays**. Cada array dentro de esta lista contiene la serie de tiempo correspondiente a un radio analizado. El orden de los arrays en la lista corresponde al orden de los radios definidos en el archivo de configuración del proyecto.

In [ ]:
from goesdl.experimental.algorithm import run_algorithm_w

bt_raw_timeseries = run_algorithm_w(settings, parameters, dataset_paths)

### 2.2 Paso 2: Tratamiento de Datos Faltantes en las Series Temporales

Un paso crítico en cualquier análisis de datos es el manejo de valores ausentes (`NaN`). En el contexto de series temporales, los datos faltantes pueden introducir sesgos, generar errores en los cálculos o afectar la calidad del análisis espectral. Esta sección aborda específicamente cómo se gestionan estos datos para asegurar la robustez de los resultados.

#### 2.2.1 Eliminación de Datos Faltantes en los Extremos de las Series

Se aplica un proceso de **recorte (`trimming`)** para eliminar cualquier valor `NaN` que pueda presentarse en los extremos (al inicio o al final) de las series temporales generadas previamente (`bt_raw_timeseries`). Los datos faltantes en los extremos son comunes debido a la irregularidad en la adquisición de datos satelitales o al inicio/fin del periodo de observación del ciclón.

La función `trim_timeseries` es la encargada de esta operación. Al eliminar estos `NaN` en los bordes, se asegura que las series temporales que se utilizarán en los pasos siguientes sean continuas y válidas en toda su extensión, lo que es esencial para la precisión de cualquier transformada de Fourier o análisis espectral. El resultado de esta operación, una serie temporal más "limpia", se almacena en la variable `bt_timeseries`.

**Motivo de este paso:** Esta eliminación de datos en los extremos es fundamental para el siguiente paso, que involucra la imputación de datos mediante **interpolación spline cúbica dispersa (`sparse cubic spline interpolation`)**. Si se intentara interpolar series con `NaN` en los bordes, al no haber datos más allá de esos límites, la extrapolación generaría **valores muy atípicos y poco realistas**, comprometiendo severamente la calidad de la serie de tiempo imputada. Recortar los extremos previene estos artefactos y asegura una interpolación más fiable.

In [ ]:
from goesdl.experimental.utilities import trim_timeseries

bt_timeseries = trim_timeseries(bt_raw_timeseries, parameters)

#### 2.2.2 Imputación de Datos Faltantes Mediante Interpolación Spline Cúbica Dispersa

Una vez eliminados los datos faltantes en los extremos de las series temporales, el siguiente paso crítico en el tratamiento de datos es la **imputación de los valores ausentes internos (`NaN`)**. Los vacíos dentro de una serie temporal, incluso después del recorte inicial, pueden afectar significativamente la precisión de los análisis subsiguientes, especialmente aquellos basados en transformadas de Fourier o que requieren una serie continua.

Para rellenar estos huecos, se utiliza un método de **interpolación spline cúbica dispersa (`sparse cubic spline interpolation`)**. Este método es ideal para datos geofísicos y series temporales, ya que permite estimar los valores faltantes de manera suave y continua, basándose en la forma general de la curva de los datos circundantes. A diferencia de interpolaciones lineales, las splines cúbicas capturan mejor la variabilidad y evitan la introducción de discontinuidades abruptas.

La función `fill_timeseries` se encarga de aplicar esta técnica de imputación. El resultado de esta operación es:
* `bt_filled_timeseries`: Las series temporales de temperatura de brillo con todos los valores faltantes imputados, lo que las hace continuas y aptas para el análisis espectral.
* `bt_gap_indices`: Un registro de los índices donde se encontraban los datos faltantes originales, lo cual puede ser útil para verificar la magnitud de la imputación.

Este paso es esencial para obtener series de tiempo completas y de alta calidad, que son fundamentales para una detección precisa y robusta del ciclo diurno.

In [ ]:
from goesdl.experimental.utilities import fill_timeseries

bt_filled_timeseries, bt_gap_indices = fill_timeseries(bt_timeseries, settings)

### 2.3 Paso 3: Submuestreo de las Series Temporales (Opcional)

Este paso se enfoca en el **submuestreo (`subsampling`)** de las series temporales de temperatura de brillo. El submuestreo implica reducir la frecuencia de muestreo de los datos, conservando solo un subconjunto de las observaciones originales. Esto puede ser útil por varias razones en el análisis de series temporales, como:
* Reducir la carga computacional para análisis posteriores.
* Alinear la resolución temporal con requisitos específicos de ciertas técnicas de análisis.
* Mitigar el ruido de alta frecuencia que podría estar presente en los datos originales.

Es importante destacar que la ejecución de este submuestreo es **condicional**. La función `subsample_timeseries` verificará los parámetros de configuración definidos en el objeto `settings` (cargados en el primer bloque del cuaderno). **Si la configuración no especifica o activa el submuestreo, la función devolverá la serie temporal de entrada (`bt_filled_timeseries`) sin ninguna modificación**, asegurando la flexibilidad y adaptabilidad del algoritmo.

El resultado de este paso es `bt_subsampled_timeseries`, que contendrá la serie temporal con la resolución temporal deseada (o la original, si no se aplicó el submuestreo).

In [ ]:
from goesdl.experimental.utilities import subsample_timeseries

bt_subsampled_timeseries = subsample_timeseries(bt_filled_timeseries, settings, parameters)

### 2.4 Paso 4: Eliminación de Tendencias de las Series Temporales

El "detrending" o eliminación de tendencias es un paso fundamental en el preprocesamiento de series temporales, especialmente cuando el objetivo es analizar componentes oscilatorias o periódicas como el ciclo diurno. Las tendencias son variaciones a largo plazo que pueden enmascarar o distorsionar las señales periódicas en el análisis espectral.

En este cuaderno, la eliminación de tendencias se realiza mediante un proceso de dos pasos para asegurar una remoción efectiva y precisa:

1.  **Sustracción de una Regresión Lineal con Término Constante (Media):** Inicialmente, se estima una línea recta mediante regresión lineal a lo largo de cada serie temporal. Esta línea recta incluye la media de la serie como su término constante. La serie de tiempo resultante de esta regresión se sustrae de la serie original, con el objetivo de eliminar la variación lineal y la media.

2.  **Sustracción Adicional de la Media Residual:** En ocasiones, debido a razones de precisión numérica o características inherentes a los datos, la sustracción de la regresión lineal no elimina completamente la media de la serie. Para asegurar que la serie resultante tenga una media de cero (o muy cercana a cero), se calcula nuevamente la media de la serie de tiempo obtenida en el primer paso y se sustrae este valor constante. Esto garantiza que no queden remanentes de la media que puedan afectar los análisis de frecuencia.

La función `detrend_timeseries` es la encargada de ejecutar este procedimiento dual. Los resultados son:
* `bt_detrended_timeseries`: Las series temporales de temperatura de brillo con la tendencia y la media eliminadas, listas para el análisis de periodicidad.
* `bt_tendency_components`: Los componentes de tendencia que fueron sustraídos (la línea recta y el valor constante residual).

Este paso es crítico para asegurar que las fluctuaciones detectadas en los pasos siguientes correspondan verdaderamente al ciclo diurno y no a variaciones de fondo o tendencias a largo plazo.

In [ ]:
from goesdl.experimental.utilities import detrend_timeseries

bt_detrended_timeseries, bt_tendency_components = detrend_timeseries(bt_subsampled_timeseries)

### 2.5 Paso 5: Cálculo de Series de Tiempo Promedio

En este paso, se calculan diferentes tipos de promedios a partir de las series temporales procesadas. Dado que `bt_detrended_timeseries` (y las series de las que deriva) son **listas de series de tiempo, cada una correspondiente a un valor de radio diferente**, el cálculo de promedios nos permite obtener una representación global del comportamiento del huracán, así como diagnosticar ciertos aspectos del procesamiento de datos.

Se computan principalmente tres tipos de promedios, cada uno con una finalidad específica:

1.  **Promedios Incoherentes de las Series Originales (con tendencia) y Detraídas (sin tendencia):**
    Estos promedios se calculan simplemente promediando las series de tiempo punto a punto, sin considerar la fase de las oscilaciones. Aunque no son ideales para extraer un ciclo promedio coherente (debido a posibles cancelaciones por desfase de las señales en distintos radios), son muy útiles para:
    * **Evaluar el grado de afectación debido a la tendencia residual:** Comparar el promedio de las series originales con el de las series sin tendencia permite visualizar la efectividad del paso de "detrending".
    * **Evaluar la cancelación por desfase:** Si las señales de un fenómeno periódico (como el ciclo diurno) tienen diferentes fases en distintos radios, al promediarlas directamente sin alineación, la señal promedio puede atenuarse o incluso desaparecer. Estos promedios incoherentes nos ayudan a observar este efecto y la necesidad de una alineación de fase para un promedio significativo.<sup>1</sup>

2.  **Promedio Coherente de las Señales Alineadas en Fase en el Dominio Frecuencial:**
    Este es el promedio más importante para caracterizar el comportamiento global del fenómeno. Para asegurar la calidad y la robustez del proceso de puesta en fase, se aplica un **prefiltrado con un filtro pasabanda Butterworth** cuyos parámetros (frecuencias de corte, ancho de banda, orden del filtro) se definen en el archivo de configuración del proyecto (`settings`). Este prefiltrado ayuda a aislar las frecuencias de interés y a mitigar el ruido antes de la alineación.


    Una vez prefiltradas, las señales se alinean en fase en el dominio de la frecuencia antes de promediarlas. Al hacer esto, las frecuencias que se repiten con una fase similar en la mayoría de los valores de radio se verán reforzadas, mientras que el ruido o las variaciones idiosincrásicas de un radio específico tienden a cancelarse. Este promedio es el que mejor representa el **comportamiento global y robusto del ciclo diurno** a través de la estructura radial del ciclón.

La función `calculate_mean_timeseries` es la encargada de generar estos promedios, entregando un diccionario `bt_mean_timeseries` que contendrá cada uno de ellos para su posterior análisis y visualización.

---
<small>1) Esta cancelación de fase ocurre frecuentemente debido al fenómeno de propagación radial del ciclo diurno, donde la señal se desplaza desde el centro hacia afuera (o viceversa) a lo largo del tiempo, haciendo que los picos y valles no coincidan temporalmente en todos los radios.</small>

In [ ]:
from goesdl.experimental.utilities import calculate_mean_timeseries

bt_mean_timeseries = calculate_mean_timeseries(bt_subsampled_timeseries, bt_detrended_timeseries, settings, parameters)

print(len(bt_mean_timeseries))
for key, value in bt_mean_timeseries.items():
    print(f"{key} : {len(value)}")

### 2.6 Paso 6: Filtrado en Banda de Interés para Visualización

Este paso final de procesamiento aplica un filtrado a las series temporales de temperatura de brillo, utilizando un filtro con características **similares al filtro pasabanda Butterworth** empleado previamente en la puesta en fase del promedio coherente. Sin embargo, la **finalidad de este filtrado es meramente visual**, y no para el análisis espectral principal.

El propósito clave es mejorar la claridad y la limpieza de las series de tiempo, lo que permite una **visualización más nítida del desfasaje entre las series de tiempo de distintos radios**. Al aislar la banda de frecuencia de interés (la del ciclo diurno), se eliminan otras fluctuaciones que podrían enmascarar los patrones de fase.

Esta visualización es de gran valor, ya que indirectamente permite:
* **Estimación cualitativa de la velocidad de propagación:** Al observar el desfase entre los picos y valles en una secuencia de radios consecutivos, se puede obtener una estimación de la velocidad con la que la "onda" de temperatura de brillo se propaga radialmente desde el centro de la tormenta hacia afuera (o viceversa).
* **Caracterización de la propagación:** Si la separación temporal de los picos/valles entre radios es uniforme, sugeriría una rapidez de propagación constante. Si no es uniforme, podría indicar una aceleración o desaceleración en la propagación de este fenómeno. Aunque el cálculo cuantitativo de la rapidez de propagación y su caracterización no se realizan explícitamente en este trabajo, este filtrado proporciona la base visual para futuras investigaciones en esa dirección.

La función `filter_timeseries` aplica este filtrado a las series detraídas (`bt_detrended_timeseries`) y a los promedios relevantes (`bt_mean_timeseries`), generando `bt_filtered_timeseries` y `bt_filtered_mean_timeseries` para las visualizaciones posteriores.

In [ ]:
from goesdl.experimental.utilities import filter_timeseries

bt_filtered_timeseries, bt_filtered_mean_timeseries = filter_timeseries(bt_detrended_timeseries, bt_mean_timeseries, settings, parameters)

### 2.7 Paso 7: Análisis Espectral Completo (Fourier) y Detección de Componentes Dominantes

Este es el núcleo analítico del cuaderno, donde se aplica un análisis espectral avanzado para identificar las periodicidades presentes en las series temporales de temperatura de brillo. La complejidad de este paso radica en la necesidad de obtener estimaciones robustas de la densidad espectral de potencia (PSD) y de la significancia estadística de los picos de frecuencia.

El proceso se desarrolla de la siguiente manera:

1.  **Análisis Espectral con Método de Welch:**
    Dependiendo de la configuración leída del archivo (`settings`), se realiza un análisis espectral utilizando el **método de Welch**. Este método es robusto para series temporales ruidosas o no estacionarias, y puede configurarse con o sin solapamiento (`overlap`) de segmentos y con un número variable de segmentos (`multiple segments`) para mejorar la estimación de la PSD. Parámetros como el tamaño de la FFT y el **tipo de ventaneo (`windowing`)** también se definen aquí, impactando la resolución y el ruido de la estimación espectral. El ventaneo es **esencial para mitigar el "spectral leakage" (fuga espectral)**, un artefacto que ocurre cuando la señal no es periódica dentro del intervalo de muestreo, dispersando la energía de una frecuencia a frecuencias adyacentes y distorsionando el espectro real. Aplicar una función de ventana reduce este efecto, mejorando la resolución y la precisión de la estimación espectral.

2.  **Filtración Topológica para Detección y Caracterización de Picos:**
    Una vez obtenida la Densidad Espectral de Potencia (PSD) para cada frecuencia, se aplica un método de **filtración topológica en 1D** al espectro. Este enfoque utiliza conceptos de conjuntos de supernivel (`superlevel sets`) y árboles de fusión (`merge trees`) para determinar de manera robusta los picos de las componentes reales de la señal.
    * Para una mayor precisión, se utilizan técnicas de **interpolación parabólica** para obtener mejores aproximaciones de la ubicación y magnitud de estos picos.
    * Adicionalmente, se realiza una filtración por conjuntos de subnivel (`sublevel sets`) para determinar los valles adyacentes a cada pico. Estos valles actúan como las **fronteras (`bounds`)** de cada pico en el espectro. Posteriormente, estas fronteras se utilizan para integrar el área bajo cada pico, lo que permite determinar la **energía total** asociada a cada componente de frecuencia dominante.

3.  **Estimación de la Curva Nula y Significancia Estadística:**
    Para evaluar la significancia estadística de los picos detectados, se estima una **curva nula (hipótesis nula)**. Esto se logra estimando los parámetros del modelo **AR(1) (Autorregresivo de orden 1)** de la serie de tiempo original. Este modelo permite crear un **modelo de ruido rojo** (que, para clarificar, es un tipo de ruido coloreado con una mayor potencia en las bajas frecuencias, a menudo presente en datos geofísicos). A partir de este modelo de ruido rojo, se pueden calcular los **p-valores** para cada frecuencia en el espectro, indicando la probabilidad de que un pico observado sea producto del azar.

Los objetos retornados por este proceso, `analysers` y `average_analysers`, son instancias de la clase `FourierAnalysis`. Esta clase encapsula todos los resultados del análisis espectral en sus miembros y provee métodos para obtener, por ejemplo, las curvas de distintos percentiles. La estimación de los grados de libertad para estos cálculos se realiza de forma automática, considerando el modelo de la ventana utilizada y los parámetros del método de Welch. Finalmente, a partir de estos análisis, se identifican explícitamente los ciclos diurnos y dominantes.

In [ ]:
from goesdl.experimental.utilities import analyze_spectra, find_diurnal_cycle, get_dominant_cycle

analysers, average_analysers = analyze_spectra(bt_detrended_timeseries, bt_mean_timeseries, settings, parameters)

diurnal_cycle, mean_diurnal_cycle = find_diurnal_cycle(analysers, average_analysers)
dominant_cycle, mean_dominant_cycle = get_dominant_cycle(analysers, average_analysers)

### 3.1 Visualización de las Series de Tiempo Originales, Imputadas y el Impacto de la Imputación

En esta subsección se presentan tres gráficos que permiten inspeccionar el estado de las series temporales y evaluar el impacto del proceso de imputación de datos faltantes:

1.  **Series Temporales Capturadas (Originales y Discontinuas):** El primer gráfico muestra las series temporales tal como fueron inicialmente capturadas y procesadas (es decir, después del cálculo de diferencias y el recorte de extremos, pero antes de la imputación). Estas curvas aparecen como **líneas discontinuas**, y en cada punto de dato medido se superponen **marcadores circulares (●)**. Este gráfico proporciona una vista de los datos crudos, permitiendo identificar visualmente las lagunas de información.

2.  **Series Temporales Imputadas (Continuas):** El segundo gráfico presenta las series temporales después de haber aplicado el proceso de imputación mediante interpolación spline cúbica dispersa. Aquí, las curvas se muestran como **líneas de trazos** y **sin marcadores**, indicando la continuidad de las series una vez que los valores faltantes han sido estimados. Este visual permite apreciar el resultado de la interpolación y la completitud de las series para análisis posteriores.

3.  **Promedio Incoherente Original con Marcadores de Imputación:** El tercer gráfico se enfoca en el **promedio incoherente de las series originales (aquellas que aún contienen la tendencia)**. Sobre esta curva, se agregan **marcadores tipo aspa ($\times$) en cada punto que fue originalmente imputado** dentro de esa serie promedio. Este gráfico es particularmente útil para **evaluar visualmente el impacto de la imputación**: permite observar si los valores imputados se integran suavemente en la serie o si introducen alguna anomalía detectable, brindando una medida cualitativa de la calidad del proceso de imputación en el promedio.

---

> **Nota sobre la Interpretación de los Gráficos:**

En todos los gráficos de series de tiempo presentados, el **eje X** representa el tiempo en **días**. El **eje Y** mide la variación en **Kelvin (K)**; es importante destacar que estos valores corresponden a **diferencias de temperatura de brillo**, no a temperaturas absolutas, por lo que pueden observarse valores tanto positivos como negativos.

Los **títulos de los gráficos** incluyen parámetros clave como la tasa de muestreo (`fs`), el margen de separación temporal para la diferencia (`dt`), y en los gráficos de series filtradas, ver § 3.3, la frecuencia central de filtrado (`fc`). Las **leyendas** de cada gráfico indican claramente a qué radio corresponde cada línea (ej., "r = 500-km").

In [ ]:
from goesdl.experimental.report import visualizar_capturas

visualizar_capturas(bt_timeseries, bt_filled_timeseries, bt_mean_timeseries, bt_gap_indices, settings, parameters)

### 3.2 Visualización de Series Temporales Detraídas (Sin Tendencia) y su Impacto en los Promedios

Esta sección se dedica a la visualización de las series temporales una vez que la tendencia ha sido eliminada. El objetivo principal es mostrar cómo el proceso de detrending afecta la forma de las series y cómo se manifiestan las diferencias entre los distintos tipos de promedios (coherentes e incoherentes) en este contexto sin tendencia. Se presentan tres gráficos clave:

1.  **Series Temporales Detraídas y Continuas:** El primer gráfico es análogo al segundo gráfico de la sección anterior (3.1), pero con una diferencia fundamental: muestra las series temporales de temperatura de brillo **después de haberles aplicado la eliminación de tendencia (`detrending`)**. Estas curvas se representan como **líneas continuas** y sin marcadores, resaltando la continuidad y la eliminación de las variaciones de largo plazo, lo que las hace más adecuadas para la detección de oscilaciones periódicas.

2.  **Comparación de Promedios Incoherentes (Con y Sin Tendencia):** El segundo gráfico superpone dos promedios:
    * El **promedio incoherente de las series sin tendencia** (en **línea continua**).
    * El **promedio incoherente de las series originales con tendencia** (en **línea de trazos**).
    Esta comparación visual es muy informativa, ya que permite **observar directamente las pendientes y las derivas** que existían en las series originales y que fueron exitosamente removidas por el proceso de detrending. Destaca la efectividad de la eliminación de tendencias en la preparación de los datos.

3.  **Comparación de Promedios Coherente e Incoherente (Sin Tendencia):** El tercer gráfico superpone el **promedio coherente** (en **línea continua**) con el **promedio incoherente de las series ya sin tendencia** (en **línea de trazos**). Este contraste es vital para **visualizar el efecto de la cancelación de fase**. Mientras que el promedio incoherente puede mostrar una señal atenuada o distorsionada debido a la propagación radial y el desfase de la onda diurna, el promedio coherente (que alinea las fases) revela la verdadera amplitud y forma del ciclo diurno global, demostrando el valor de la técnica de alineación de fase.

---

Para realizar un análisis espectral correcto, debemos eliminar toda tendencia que pueda estar afectando a las series de tiempo. Estas tendencias pueden ser interpretadas erróneamente como oscilaciones de muy baja frecuencia de gran amplitud. En los ciclones, las tendencias pueden estar generadas por características del ciclo de vida del fenómeno.

Para este estudio, asumimos una tendencia lineal y la determinamos haciendo una regresión lineal a cada serie $S(t)_r$. La línea resultante, $y(t)_r = mt + b$, se substrae de la serie resultando una serie de tiempo sin tendencia lineal, $\hat S(t)_r = S(t)_r - y(t)_r$. El término constante $b$ es la media de la serie de tiempo.

Los promedios incoherentes se calculan, simplemente, promediando todas las series de tiempo capturadas: $\bar S(t) = \frac{1}{N} \sum_{r=r_\text{m{\'i}n}}^{r_\text{m\'ax}} S(t)_r$, e igualmente con las series sin tendencia.

Se procede análogamente para obtener el promedio coherente, poniendo antes todas las series en fase para reducir la cancelación debido a la propagación de fenómeno.

Los promedios representarían el comportamiento medio del fenómeno a lo largo de los radios estudiados.

In [ ]:
from goesdl.experimental.report import visualizar_normalizados

visualizar_normalizados(bt_detrended_timeseries, bt_mean_timeseries, settings, parameters)

### 3.3 Visualización de Series Temporales Filtradas y su Relevancia para el Análisis del Ciclo Diurno

Esta sección presenta las series temporales de temperatura de brillo después de haber sido filtradas en la banda de interés del ciclo diurno (como se detalló en el Paso 6, § 2.6). La finalidad de estas visualizaciones es permitir una inspección clara de las oscilaciones diarias y extraer conclusiones fundamentales sobre el comportamiento del huracán.

Los gráficos permiten observar la **contribución de cada serie de tiempo radial a la frecuencia central** (el ciclo diurno) que se está analizando.
* **Amplitud y Evidencia del Ciclo:** Una mayor amplitud en las oscilaciones de una serie filtrada indica un ciclo diurno más intenso y evidente en la región radial correspondiente.

Se prestan especial atención a los promedios:

* **Promedio Coherente: Evidencia del Ciclo de Vida del Ciclón:**
    El promedio coherente, obtenido al promediar todas las series de tiempo radiales previamente alineadas en fase (lo que reduce la cancelación debida a la propagación del fenómeno), es un indicador robusto del comportamiento global del ciclo diurno. En este gráfico, se pueden observar indicios tanto del **fortalecimiento (amplificación)** como de la **extinción (atenuación)** del fenómeno a medida que transcurre el tiempo, evidenciando de esta manera el **ciclo de vida general del ciclón tropical** en relación con su modulación diurna.

* **Promedio Incoherente: Revelando la Propagación Radial:**
    El promedio incoherente, por su parte, es invaluable para **visualizar el efecto de la cancelación de fase**, que a su vez evidencia la **propagación radial del fenómeno con una rapidez finita**. Al comparar las fases de los picos y valles en las series de radio consecutivas, es posible **estimar visualmente la rapidez de esta propagación**. Si la separación temporal entre los picos (o valles) es uniforme a lo largo del radio, indicaría una rapidez de propagación constante; de lo contrario, se podría inferir una aceleración o desaceleración. En el caso particular del **Huracán Iván de 2004**, las observaciones preliminares en estos gráficos sugieren que la **rapidez de propagación de esta onda de temperatura de brillo aumenta con el radio**.

In [ ]:
from goesdl.experimental.report import visualizar_filtrados

visualizar_filtrados(bt_filtered_timeseries, bt_filtered_mean_timeseries, settings, parameters)

### 3.4 Visualización de los Resultados del Análisis Espectral (Periodogramas y P-valores)

Esta sección es crucial para la interpretación de los resultados del análisis espectral. La función `visualizar_espectrogramas` genera un conjunto de tres componentes de visualización **para cada una de las series radiales estudiadas**, así como para los promedios globales, permitiendo una inspección exhaustiva de las frecuencias dominantes y su significancia estadística.

Cada bloque de visualización para un radio específico (o promedio) consta de:

1.  **Interpretación Automática de P-valores Espectrales:**
    Se genera un resumen textual detallado que proporciona una interpretación cuantitativa y cualitativa de los resultados del análisis de significancia estadística. Este texto ofrece una visión rápida sobre:
    * **Estadísticas Generales:** Número total de frecuencias analizadas, frecuencias puras preseleccionadas, y los valores mínimos/medianos de p-valor encontrados.
    * **Niveles de Significancia:** Recuento y porcentaje de frecuencias que superan los umbrales de p < 0.05, p < 0.01 y p < 0.001.
    * **Evaluación Estadística:** Una conclusión sumaria sobre la actividad espectral (ej., "Espectro muy silencioso") en comparación con lo esperado.
    * **Resumen Ejecutivo:** Una categorización general de la actividad del espectro (ej., "Baja actividad").
    * **Frecuencias Significativas:** Listado explícito de las frecuencias con los p-valores más bajos, clasificadas por niveles de significancia ("Muy Significativas" para p < 0.01 y "Significativas" para p < 0.05), mostrando su valor en ciclos por día (c/d), periodo en horas por ciclo (h/c) y el p-valor exacto.
    * **Advertencias y Recomendaciones:** Consejos para la interpretación o posibles mejoras en el análisis (ej., verificar el modelo de ruido, aumentar la resolución espectral).

2.  **Periodograma (Gráfico de Densidad Espectral de Potencia - PSD):**
    Este gráfico es la representación visual de la energía de las diferentes frecuencias presentes en la serie. Para cada radio, se muestra:
    * La **PSD local** de la serie de tiempo del radio actual (línea continua azul).
    * La **PSD global** del promedio coherente (línea discontinua "`-.-`" naranja), permitiendo comparar el comportamiento local con el promedio general.
    * Las **líneas de hipótesis nula** (gris punteada) y los **percentiles 90, 95 y 99** (azul, verde y rojo carmesí punteadas, respectivamente). Estas líneas son cruciales para evaluar la significancia de los picos de potencia.
    * **En el título de cada periodograma se incluyen detalles exhaustivos para una referencia rápida, como:** el radio (`r`) bajo análisis; la frecuencia (`f`), el periodo (`T`) y el p-valor (`p`) del pico dominante; el modelo de ruido utilizado (`rojo AR(1)`), y los grados de libertad (`dof`).
    * **Marcadores:** Se incluyen marcadores verticales en la grilla para las frecuencias **diurna (1 c/d)** y **semidiurna (2 c/d)**. Todos los picos locales que superan el percentil 95 son etiquetados con su frecuencia y se marcan con cruces (`+`) azules si superan el percentil 95, y verdes si superan el 99. Además, el pico dominante global en la curva global se indica con un marcador de cruz (`+`) rojo.
    * **Segmentos Significativos:** El segmento de la PSD local que supera el percentil 95 se resalta visualmente, pintándose de color cian, lo que facilita la identificación de las bandas de frecuencia con mayor significancia.
    * El **eje X** mide la **frecuencia en ciclos por día (c/d)**.
    * El **eje Y** representa la **Densidad Espectral de Potencia (PSD)** en unidades de $K^2 \cdot d/c$.

3.  **Gráfico de $p$-valores:**
    Este gráfico complementa el periodograma mostrando la significancia estadística de cada frecuencia de manera directa:
    * La **curva de $p$-valores** se grafica como una línea continua.
    * Se superponen líneas horizontales para los umbrales de significancia **$\alpha = 0.05$** (línea punteada amarilla, indicando "significativo"), **$\alpha = 0.01$** (línea discontinua "`-.-`" naranja, para "muy significativo") y **$\alpha = 0.001$** (línea de trazos roja, para "extremadamente significativo").
    * Las regiones del fondo del gráfico se pintan en colores suaves similares para indicar las **zonas de evidencia moderada, fuerte y extrema**, respectivamente, haciendo visualmente intuitiva la identificación de las frecuencias estadísticamente significativas.
    * El **eje X** mide la **frecuencia en ciclos por día (c/d)**.
    * El **eje Y** representa el **$p$-valor**.

In [ ]:
from goesdl.experimental.report import visualizar_espectrogramas

visualizar_espectrogramas(analysers, average_analysers, settings, parameters)

### 3.5 Visualización de los Ciclos Diurnos Detectados en Cada Serie de Tiempo Radial

Esta sección final presenta una visualización clave que integra los resultados de los análisis previos. El objetivo es mostrar el ciclo diurno detectado en el contexto de las series temporales individuales, permitiendo una inspección directa de cómo la periodicidad identificada se alinea con las fluctuaciones de los datos.

Para cada radio analizado, se genera un gráfico que incluye:

* **Serie de Tiempo Original (Detraída):** Representada por una línea continua de color celeste. Aunque es la "original" para esta visualización, es importante recordar que esta serie ya ha pasado por el detrending y la imputación, sirviendo como la base de datos limpia sobre la que se realizan los análisis.
* **Serie de Tiempo Filtrada:** Mostrada como una línea de puntos y trazos (punto-guion) de color naranja. Esta es la serie después de aplicar el filtro pasabanda para aislar la frecuencia del ciclo diurno, permitiendo una visión más clara de la oscilación principal.
* **Ciclo Diurno Detectado:** Se grafica como una línea de trazos (dashed) de color rojo. Es la representación de la componente periódica del ciclo diurno extraída del análisis espectral. Para mejorar su visibilidad y permitir una comparación más clara con las series originales y filtradas, su **amplitud real ha sido multiplicada por tres**.
* **Eje X:** Se han incorporado las fechas reales y las horas espaciadas cada 6 horas, en la escala UTC, proporcionando un contexto temporal preciso para la interpretación de las oscilaciones.

Estos gráficos permiten una comparación directa entre los datos crudos (pero procesados), la señal filtrada y el ciclo diurno modelado, ofreciendo una validación visual de la detección y caracterización del fenómeno.

In [ ]:
from goesdl.experimental.report import visualizar_ciclos_dominantes

visualizar_ciclos_dominantes(
    diurnal_cycle, mean_diurnal_cycle, dominant_cycle, mean_dominant_cycle, analysers,
    average_analysers, bt_detrended_timeseries, bt_filtered_timeseries, bt_mean_timeseries,
    bt_filtered_mean_timeseries, settings, parameters
)

### 3.6 Sobre la Interpretación y Difusión de los Resultados

Un ***periodograma*** o *espectro de potencia*, muestra la distribución de la **densidad espectral de potencia** de una serie de tiempo en función de la **frecuencia**.

La **densidad espectral de potencia** de una señal es una función matemática que nos informa de cómo está distribuida la potencia de dicha señal sobre las distintas frecuencias de las que está formada.

> Aunque la densidad espectral no es exactamente lo mismo que el espectro de una señal, a veces ambos términos se usan indistintamente, lo cual, en rigor, es incorrecto.

#### 3.6.1 Cómo interpretar los resultados

**Picos significativos**: Cuando la potencia espectral supera la línea del percentil, $\rho$, correspondiente, sugiere que esa frecuencia contiene señal real, no solo ruido, con nivel de confianza estadística $100 \cdot (1 - \alpha)\%$ o, dicho de otra manera, con una significancia igual $100 \cdot \alpha\%$. $\rho$ es una medida de cuán confiados queremos estar.

El **valor** $p$, o $p$-valor, es una estadístico que nos muestra la probabilidad de haber obtenido el resultado que hemos logrado suponiendo que la hipótesis nula, $H_0$, es cierta. Valores altos de $p$ no permiten rechazar $H_0$, mientras que valores bajos de $p$ sí permiten rechazar $H_0$.

En una prueba estadística, se rechaza la hipótesis nula $H_0$ si el valor $p$ asociado al resultado observado es igual o menor que un nivel de significación $\alpha$ establecido arbitrariamente, convencionalmente $0.05$ o $0.01$. En otras palabras, si el resultado obtenido es más inusual que el rango esperado de resultados dada una hipótesis nula $H_0$ cierta y el nivel de significación $\alpha$ elegido, es decir si $p$ es menor que $\alpha$, podemos decir que tenemos un resultado estadísticamente significativo que permite rechazar $H_0$.

Es importante recalcar que un contraste de hipótesis no permite **aceptar** una hipótesis; simplemente la rechaza o no la rechaza, es decir que la tacha de verosímil (lo que no significa obligatoriamente que sea cierta, simplemente que es más probable de serlo) o inverosímil.

> *La significación estadística de un resultado no implica que el resultado también tenga relevancia en el mundo real. Por ejemplo, un efecto estadísticamente significativo puede ser demasiado pequeño para ser interesante.*

##### Evaluación práctica:

* Picos por encima del percentil $95\%$ son estadísticamente significativos
* Picos entre percentiles $90\text{--}95\%$ son marginalmente significativos
* Picos por debajo del $90\%$ pueden ser ruido

Picos alrededor y por debajo de la línea nula

* **No significa**: "Definitivamente es ruido", "No hay información útil ahí"
* **Sí significa**: "Comportamiento consistente con variabilidad natural del ruido", "No hay evidencia de señal determinística fuerte"

Un pico por debajo de la línea nula tiene **alta probabilidad de ser una fluctuación aleatoria** del proceso de ruido, pero esta probabilidad no es abrumadoramente alta como en el caso de picos muy por encima de percentiles altos. Es una **región de "ruido típico"** donde no se pueden hacer afirmaciones fuertes en ninguna dirección.

##### Ruido Rojo (Red Noise)

El ruido rojo representa un proceso autoregresivo de primer orden AR(1), caracterizado por:

* Correlación positiva entre valores consecutivos
* Mayor potencia en frecuencias bajas que decae hacia altas frecuencias
* Los umbrales de ruido rojo son generalmente más altos que los de ruido blanco
* Es común en sistemas meteorológicos, climáticos y geofísicos debido a la inercia natural
* El ruido rojo es más realista para datos naturales con memoria temporal

##### Interpretación de percentiles

* La línea nula asume ruido rojo (comportamiento típico, sin correlación temporal), se denomina así porque representa la hipótesis nula, $H_0$.
* Solo el $5\%$ del espectro de ruido rojo excederá la línea de percentil $95\%$ por casualidad (probablemente señal real).
* Solo el $1\%$ excederá la línea de percentil $99\%$ aleatoriamente (muy probablemente señal real).
* El intervalo de confianza del $100 \cdot (1 - \alpha)\%$ es toda la región que se encuentra por encima de la línea de percentil $100\cdot(1 - \alpha)\%$.

Esta metodología es especialmente importante en meteorología, climatología, oceanografía y geofísica, donde distinguir señales reales de variabilidad natural es fundamental para identificar ciclos, tendencias o periodicidades genuinas en los datos.

#### 3.6.2 Prueba de hipótesis

En el contexto de pruebas de significancia espectral se trata de una prueba de cola derecha unilateral.

* $H_0(f)$: El pico de potencia de densidad espectral, observado en la frecuencia $f$, proviene únicamente del proceso de ruido estocástico de fondo.
* $H_1(f)$: El pico de potencia de densidad espectral observado contiene señal determinística real en la frecuencia $f$.

Rechazar $H_0(f)$ implica que existe una componente de señal determinística en la frecuencia $f$.

##### Test de Hipótesis Formal

* **Estadístico de prueba**: $T(f) = S_{\text{obs}}(f) / S_{\text{nulo}}(f)$
* **Distribución nula**: $T(f) \sim F(2, \infty)$ bajo $H_0$
* **Decisión**: Rechazar $H_0$ si $T(f) > F_{1 - \alpha}(2, \infty)$

donde $S_{\star}(f)$ es la potencia de la densidad espectral. En análisis espectral, la formulación correcta de $H_0$ debe ser específica por frecuencia $f$, no global.

##### Razones Metodológicas

1. **Independencia Estadística**: Cada frecuencia constituye una prueba de hipótesis independiente.
2. **Distribución Espectral del Ruido**: La función del modelo nulo AR(1) varía con la frecuencia, por lo que $H_0$ debe evaluarse punto a punto.
3. **Control de Error Tipo I**: Una formulación global crearía problemas de *comparaciones múltiples*:
    * Si evaluamos $N$ frecuencias con $\alpha = 0.05$ cada una.
    * Probabilidad de al menos un falso positivo $\approx 1 - (1 - 0.05)^N$.

##### Formulación Matemática Precisa

Para cada bandeja espectral $f_i$:

* $H_0(f_i)$: $S_{\text{obs}}(f_i) \sim S_{\text{nulo}}(f_i) \cdot \chi^2(2) / 2$,
* $H_1(f_i)$: $S_{\text{obs}}(f_i) = S_{\text{se\~nal}}(f_i) + S_{\text{nulo}}(f_i) \cdot \chi^2(2) / 2$,

donde $S_{\text{se\~nal}}(f_i) > 0$ representa la componente determinística.

#### 3.6.3 $p$-valor (Probabilidad de error si declaro significativo)

> Curva de $p$-valores: ¿Qué tan improbable es cada pico si fuera solo ruido?

##### Cómo explicarlo a otras personas

###### Lenguaje simple:

*"Este gráfico muestra qué tan sorprendente es cada pico del espectro. Valores bajos (cerca del fondo) significan 'muy sorprendente si fuera solo ruido', valores altos (cerca del tope) significan 'normal, probablemente ruido'."*

###### Analogía efectiva:

"Imagina que estás escuchando una orquesta tocando. El gráfico de $p$-valores te dice, para cada nota musical, qué tan probable es que sea solo el ruido de fondo del auditorio versus una nota real de los instrumentos. Valores muy bajos = definitivamente música real."

##### Guía de interpretación rápida del gráfico de $p$-valores

###### Para Audiencias Técnicas:

"Los $p$-valores cuantifican la probabilidad de que cada pico espectral sea una fluctuación aleatoria. Valores $< 0.05$ sugieren señales reales con $95\%$ de confianza."

###### Para Audiencias Generales:

"Este gráfico identifica qué partes de la señal son 'música real' versus 'ruido de fondo'. Las líneas que tocan el fondo del gráfico representan señales genuinas."

##### Puntos clave para presentaciones:

1. Escala logarítmica es esencial: Diferencias entre 0.001 y 0.01 son enormes.
2. Distribución uniforme = solo ruido: Si los $p$-valores están esparcidos uniformemente.
3. Valles profundos = señales: Concentraciones de $p$-valores muy bajos.
4. Contexto físico importa: Un $p = 0.04$ puede ser ruido, un $p = 10^{-6}$ definitivamente no.

##### Frases Útiles para Explicar:

* *"Mientras más bajo el valle, más confiamos en que hay señal real"*
* *"La línea roja es nuestro umbral de confianza del $\,95\%$"*
* *"Si fuera solo ruido, esperaríamos ver una línea plana y ondulada"*

El gráfico de $p$-valores es tu "detector de señales cuantitativo" - te dice exactamente qué tan seguros puedes estar de cada componente espectral.

#### 3.6.4 Conclusión

La línea nula representa el **comportamiento típico del azar**, conceptualmente es el **centro de la distribución del azar**. Es la referencia central contra la cual medimos qué tan improbable es observar desviaciones hacia arriba, las líneas de percentiles.

Al afirmar que **un pico que sobrepasa el percentil $100 \cdot (1 - \alpha)\%$ proviene de un proceso natural (señal real)**, existe una probabilidad de $100 \cdot \alpha\%$ de estar cometiendo un error, o también, la probabilidad de $100 \cdot \alpha\%$ de que lo que se esté afirmando sea falso. Pero la afirmación es **estadísticamente correcta**.

Esta es la interpretación clásica y correcta del $p$-valor en análisis espectral.

> **Y recuerden amigos**: *cuanto más valle, mejor...*, ¡vaamonooo!